In [10]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("D:/adaptive_federated_gnn")

TRAIN_CSV = BASE / "D:/adaptive_federated_gnn/data/raw/other datasets/GeneratedLabelledFlows/TrafficLabelling/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv"
EDGE_TABLE = BASE / "data/processed/edge_table.csv"
NEW_CSV = BASE / "D:/adaptive_federated_gnn/data/raw/other datasets/GeneratedLabelledFlows/TrafficLabelling/Friday-02-03-2018_TrafficForML_CICFlowMeter.csv"

OUT = BASE / "outputs/day23"
OUT.mkdir(parents=True, exist_ok=True)

print("Original training CSV exists:", TRAIN_CSV.exists())
print("Processed edge table exists:", EDGE_TABLE.exists())
print("New CIC-IDS2018 CSV exists:", NEW_CSV.exists())

if not NEW_CSV.exists():
    raise FileNotFoundError(
        "Download the CIC-IDS2018 CSV and save it to:\n"
        f"{NEW_CSV}"
    )

Original training CSV exists: True
Processed edge table exists: True
New CIC-IDS2018 CSV exists: True


In [11]:
old_columns = pd.read_csv(TRAIN_CSV, nrows=5, low_memory=False).columns
edge_columns = pd.read_csv(EDGE_TABLE, nrows=5, low_memory=False).columns
new_columns = pd.read_csv(NEW_CSV, nrows=5, low_memory=False).columns

def clean_columns(columns):
    return [str(c).strip() for c in columns]

old_columns = clean_columns(old_columns)
edge_columns = clean_columns(edge_columns)
new_columns = clean_columns(new_columns)

print("Original CIC-IDS2017 columns:", len(old_columns))
print("Processed edge-table columns:", len(edge_columns))
print("New CIC-IDS2018 columns:", len(new_columns))

print("\nCIC-IDS2018 sample columns:")
print(new_columns[:20])

print("\nCIC-IDS2018 final columns:")
print(new_columns[-10:])

Original CIC-IDS2017 columns: 85
Processed edge-table columns: 11
New CIC-IDS2018 columns: 80

CIC-IDS2018 sample columns:
['Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std']

CIC-IDS2018 final columns:
['Fwd Seg Size Min', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min', 'Label']


In [12]:
# Show columns that appear in both datasets
common_columns = sorted(set(old_columns) & set(new_columns))

# Show differences between the two raw schemas
only_old = sorted(set(old_columns) - set(new_columns))
only_new = sorted(set(new_columns) - set(old_columns))

print("Common column names:", len(common_columns))
print("Only in CIC-IDS2017:", len(only_old))
print("Only in CIC-IDS2018:", len(only_new))

print("\nCommon columns (first 30):")
print(common_columns[:30])

print("\nOnly in CIC-IDS2017 (first 20):")
print(only_old[:20])

print("\nOnly in CIC-IDS2018 (first 20):")
print(only_new[:20])

Common column names: 30
Only in CIC-IDS2017: 55
Only in CIC-IDS2018: 50

Common columns (first 30):
['Active Max', 'Active Mean', 'Active Min', 'Active Std', 'Bwd IAT Max', 'Bwd IAT Mean', 'Bwd IAT Min', 'Bwd IAT Std', 'Bwd PSH Flags', 'Bwd URG Flags', 'CWE Flag Count', 'Down/Up Ratio', 'Flow Duration', 'Flow IAT Max', 'Flow IAT Mean', 'Flow IAT Min', 'Flow IAT Std', 'Fwd IAT Max', 'Fwd IAT Mean', 'Fwd IAT Min', 'Fwd IAT Std', 'Fwd PSH Flags', 'Fwd URG Flags', 'Idle Max', 'Idle Mean', 'Idle Min', 'Idle Std', 'Label', 'Protocol', 'Timestamp']

Only in CIC-IDS2017 (first 20):
['ACK Flag Count', 'Average Packet Size', 'Avg Bwd Segment Size', 'Avg Fwd Segment Size', 'Bwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Header Length', 'Bwd IAT Total', 'Bwd Packet Length Max', 'Bwd Packet Length Mean', 'Bwd Packet Length Min', 'Bwd Packet Length Std', 'Bwd Packets/s', 'Destination IP', 'Destination Port', 'ECE Flag Count', 'FIN Flag Count', 'Flow Bytes/s', 'Flow ID']

Only

In [13]:
# Identify likely metadata and label columns
metadata_words = [
    "label", "source ip", "destination ip", "src ip", "dst ip",
    "flow id", "timestamp"
]

def is_metadata(col):
    name = col.lower().strip()
    return any(word in name for word in metadata_words)

# Candidate features from your existing processed edge table
model_feature_candidates = [
    col for col in edge_columns
    if not is_metadata(col)
]

new_feature_set = set(new_columns)

available = [
    col for col in model_feature_candidates
    if col in new_feature_set
]

missing = [
    col for col in model_feature_candidates
    if col not in new_feature_set
]

print("Candidate existing features:", len(model_feature_candidates))
print("Matching names in CIC-IDS2018:", len(available))
print("Names missing from CIC-IDS2018:", len(missing))

print("\nMatching feature names:")
print(available[:40])

print("\nMissing feature names:")
print(missing[:40])

compatibility_df = pd.DataFrame({
    "existing_feature": model_feature_candidates,
    "available_in_cicids2018": [
        col in new_feature_set for col in model_feature_candidates
    ]
})

compatibility_df.to_csv(
    OUT / "feature_compatibility.csv", index=False
)

print("\nReport saved:", OUT / "feature_compatibility.csv")

Candidate existing features: 5
Matching names in CIC-IDS2018: 1
Names missing from CIC-IDS2018: 4

Matching feature names:
['Protocol']

Missing feature names:
['Source Port', 'Destination Port', 'source_node', 'destination_node']

Report saved: D:\adaptive_federated_gnn\outputs\day23\feature_compatibility.csv


In [14]:
from pathlib import Path
import torch
import pandas as pd

BASE = Path("D:/adaptive_federated_gnn")

GRAPH_PATH = BASE / "data/processed/edge_graph.pt"
EDGE_TABLE = BASE / "data/processed/edge_table.csv"

graph = torch.load(
    GRAPH_PATH,
    map_location="cpu",
    weights_only=False
)

print("Graph keys:", list(graph.keys()))
print("Edge feature shape:", tuple(graph["edge_attr"].shape))
print("Edge index shape:", tuple(graph["edge_index"].shape))
print("Edge label shape:", tuple(graph["edge_label"].shape))

edge_table = pd.read_csv(EDGE_TABLE, nrows=5)

print("\nProcessed edge-table columns:")
print(edge_table.columns.tolist())

print("\nProcessed edge-table preview:")
display(edge_table.head())

Graph keys: ['edge_index', 'edge_attr', 'edge_label']
Edge feature shape: (286095, 67)
Edge index shape: (2, 286095)
Edge label shape: (286095,)

Processed edge-table columns:
['Flow ID', 'Source IP', 'Destination IP', 'Source Port', 'Destination Port', 'Protocol', 'Timestamp', 'Label', 'Binary_Label', 'source_node', 'destination_node']

Processed edge-table preview:


,Flow ID,Source IP,Destination IP,Source Port,Destination Port,Protocol,Timestamp,Label,Binary_Label,source_node,destination_node
0,192.168.10.12-192.168.10.50-35396-22-6,192.168.10.12,192.168.10.50,35396,22,6,7/7/2017 1:00,BENIGN,0,0,2
1,192.168.10.16-192.168.10.50-60058-22-6,192.168.10.16,192.168.10.50,60058,22,6,7/7/2017 1:00,BENIGN,0,1,2
2,192.168.10.12-192.168.10.50-35396-22-6,192.168.10.12,192.168.10.50,35396,22,6,7/7/2017 1:00,BENIGN,0,0,2
3,192.168.10.12-192.168.10.50-35398-22-6,192.168.10.12,192.168.10.50,35398,22,6,7/7/2017 1:00,BENIGN,0,0,2
4,192.168.10.12-192.168.10.50-35396-22-6,192.168.10.50,192.168.10.12,22,35396,6,7/7/2017 1:00,BENIGN,0,2,0


In [15]:
from pathlib import Path

BASE = Path("D:/adaptive_federated_gnn")

keywords = ["edge_attr", "edge_features", "feature_columns"]

for notebook_path in (BASE / "notebooks").glob("*.ipynb"):
    try:
        import json
        notebook = json.loads(notebook_path.read_text(encoding="utf-8"))

        for cell_number, cell in enumerate(notebook.get("cells", []), start=1):
            source = "".join(cell.get("source", []))

            if any(keyword in source for keyword in keywords):
                print(f"\n{'=' * 70}")
                print(f"Notebook: {notebook_path.name} | Cell: {cell_number}")
                print(source[:4000])

    except Exception as error:
        print(f"Could not read {notebook_path.name}: {error}")


Notebook: day10_federated_clients.ipynb | Cell: 3
edge_graph = torch.load(
    GRAPH_PATH,
    map_location="cpu",
    weights_only=False
)

edge_table = pd.read_csv(EDGE_TABLE_PATH)

edge_index = edge_graph["edge_index"]
edge_features = edge_graph["edge_attr"]
edge_labels = edge_graph["edge_label"]

print("Number of nodes:", edge_graph["edge_index"].max().item() + 1)
print("Number of edges:", edge_index.shape[1])
print("Number of edge features:", edge_features.shape[1])

print("\nEdge table shape:", edge_table.shape)
print("\nLabel distribution:")
print(edge_labels.bincount())

Notebook: day10_federated_clients.ipynb | Cell: 11
client_graphs = {}

for client_name, indices in clients.items():
    indices_tensor = torch.tensor(
        indices,
        dtype=torch.long
    )

    client_graphs[client_name] = {
        "edge_index": edge_index,
        "edge_features": edge_features[indices_tensor],
        "edge_labels": edge_labels[indices_tensor],
        "local_edge_indices": indice

In [16]:

from pathlib import Path
import json

BASE = Path("D:/adaptive_federated_gnn")

keywords = [
    "edge_attr =",
    "edge_features =",
    "torch.tensor(",
    "StandardScaler",
    "feature_columns",
    "drop(columns",
    "select_dtypes",
    "to_numpy()",
    "values.astype"
]

for folder in ["notebooks", "src"]:
    root = BASE / folder

    if not root.exists():
        continue

    for path in root.rglob("*"):
        if path.suffix == ".ipynb":
            try:
                notebook = json.loads(
                    path.read_text(encoding="utf-8")
                )
                cells = notebook.get("cells", [])

                for i, cell in enumerate(cells, start=1):
                    source = "".join(cell.get("source", []))

                    if any(k in source for k in keywords):
                        print(f"\n--- {path.name} | Cell {i} ---")
                        print(source[:2500])

            except Exception as e:
                print("Could not read:", path.name, e)

        elif path.suffix == ".py":
            try:
                source = path.read_text(encoding="utf-8")

                if any(k in source for k in keywords):
                    print(f"\n--- {path} ---")
                    print(source[:4000])

            except Exception as e:
                print("Could not read:", path, e)



--- day10_federated_clients.ipynb | Cell 3 ---
edge_graph = torch.load(
    GRAPH_PATH,
    map_location="cpu",
    weights_only=False
)

edge_table = pd.read_csv(EDGE_TABLE_PATH)

edge_index = edge_graph["edge_index"]
edge_features = edge_graph["edge_attr"]
edge_labels = edge_graph["edge_label"]

print("Number of nodes:", edge_graph["edge_index"].max().item() + 1)
print("Number of edges:", edge_index.shape[1])
print("Number of edge features:", edge_features.shape[1])

print("\nEdge table shape:", edge_table.shape)
print("\nLabel distribution:")
print(edge_labels.bincount())

--- day10_federated_clients.ipynb | Cell 11 ---
client_graphs = {}

for client_name, indices in clients.items():
    indices_tensor = torch.tensor(
        indices,
        dtype=torch.long
    )

    client_graphs[client_name] = {
        "edge_index": edge_index,
        "edge_features": edge_features[indices_tensor],
        "edge_labels": edge_labels[indices_tensor],
        "local_edge_indices": indices_tens